# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library. The dataset is described by a Croissant schema, and all references to fields, record sets, and columns use their `@id` as per the schema specification.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Access metadata fields
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Keywords: {metadata.keywords}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"Identifier (DOI): {metadata.identifier}")

## 2. Data Overview
Examine available record sets, their fields and column `@id`s, for programmatic data access. All entity references use their `@id` as defined in the Croissant schema.

Let's list the available record sets, and for each, show their fields (columns) by `@id`.

In [ ]:
# List all record sets
record_sets = dataset.record_sets # This gives a dict mapping `@id` to RecordSet objects

print("Record Sets in the Dataset:\n------------------------------")
for rs_id, rs_obj in record_sets.items():
    print(f"@id: {rs_id}\n  Name: {getattr(rs_obj, 'name', None)}\n  Description: {getattr(rs_obj, 'description', None)}")
    # Print fields (columns) for this record set
    if hasattr(rs_obj, 'fields') and rs_obj.fields:
        print("  Fields/Columns:")
        for f in rs_obj.fields:
            print(f"    - {f['@id']} (name: {f.get('name')}, dataType: {f.get('dataType')})")
    print()

## 3. Data Extraction
Load one or more record sets into pandas DataFrames for analysis.

We recommend choosing the main survey records set for primary analysis. In this dataset, the main record set appears to be `cr:survey_responses` (see the Croissant JSON or use the cell above to confirm the correct `@id`).
Replace with the actual `@id` if different. If there are multiple record sets you care about, add their `@id`s to the `record_set_ids` list.

In [ ]:
# Example: Replace these with the actual survey record set @id(s) printed above.
# For demonstration, we'll try common candidates; adjust as needed if actual @id is different.

record_set_ids = ['cr:survey_responses'] # Add more @ids if there are multiple record sets
dataframes = {}

# Load each record set into a dataframe
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set: {record_set_id}")
        print("Columns:", dataframes[record_set_id].columns.tolist())
    except Exception as e:
        print(f"Could not load '{record_set_id}': {e}")

# Show head of the main record set
df_main = dataframes[record_set_ids[0]]
df_main.head()

## 4. Exploratory Data Analysis (EDA)
Let's perform basic EDA: filtering, normalization, and grouping. We'll use the GAD-7 score field as an example for numeric analysis.

First, confirm the exact `@id` (column name) for the GAD-7 total score, e.g. `cr:gad7_total`. You can find this in the columns printed in the previous step.

In [ ]:
# Example field IDs (customize these if you see others in your schema)
numeric_field = 'cr:gad7_total'  # Use actual @id from columns
main_rs = record_set_ids[0]

# Sometimes numeric columns are loaded as strings, try to convert
df_main[numeric_field] = pd.to_numeric(df_main[numeric_field], errors='coerce')

# Remove outliers: keep scores between 0 and 21 (valid for GAD-7)
score_min, score_max = 0, 21
filtered_df = df_main[(df_main[numeric_field] >= score_min) & (df_main[numeric_field] <= score_max)]
print(f"Filtered {numeric_field} between {score_min} and {score_max}, N={len(filtered_df)}")
print(filtered_df[[numeric_field]].head())

# Normalize the scores
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} (mean 0, std 1):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Example group by: gender, if available
group_field = 'cr:gender' # Use actual @id for gender from your column list
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Mean {numeric_field} grouped by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of GAD-7 scores and (if available) compare by gender.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of GAD-7 scores
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field].dropna(), bins=range(score_min, score_max+2), kde=True)
plt.title("GAD-7 Score Distribution")
plt.xlabel("GAD-7 Total Score")
plt.ylabel("Frequency")
plt.show()

# Boxplot by gender (if available)
if group_field in filtered_df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to use `mlcroissant` to programmatically explore and process the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya. 

Key takeaways:
- The dataset exposes structured survey records and metadata through a standards-based Croissant schema, making programmatic access robust and reproducible.
- All programmatic accesses referenced fields and record sets using their `@id`.
- We showed how to extract, filter, normalize, and visualize survey data (e.g., GAD-7 scores) for further analysis.

Review the Croissant schema for more advanced usage, such as linking to related assessment fields (PHQ-9, PSQ), applying privacy filters, or integrating with other datasets.

See https://sen.science/doi/10.71728/senscience.vcs2-05nj for more details and data documentation.